In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# load the packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import glob
import gc
import re

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt
import re

# Silence the Pandas future warning
pd.set_option('future.no_silent_downcasting', True)

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, 'Cluster_Time_Series_Activation')
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 30
DT = 1.0 / FPS
DOWNSAMPLE_RATE = 15 # 2 Hz (Every 0.5 seconds)
N_CLUSTERS = 6
MAX_VELOCITY = 38.28
MAX_ACCELERATION = 16.40
MAX_DECELERATION = -50.0

ACTIVE_CLUSTERS = [0, 1, 3, 5]

def extract_period_number(p_val):
    """Safely extracts integer period, converting 'OT' to 4"""
    val_str = str(p_val).upper().strip()
    if val_str == 'OT' or val_str == 'POT':
        return 4
    try:
        match = re.search(r'\d+', val_str)
        return int(match.group()) if match else 1
    except:
        return 1

# ==========================================
# 1. TIME-SERIES FEATURE ENGINEERING
# ==========================================
def extract_time_series_features(tracking_file):
    df_raw = pd.read_csv(tracking_file, low_memory=False)

    filename = os.path.basename(tracking_file)
    game_date = filename.split('.')[0]

    df_raw['Frame_Number'] = df_raw['Image Id'].str.split('_').str[-1].astype(int)
    df_raw = df_raw.rename(columns={'Rink Location X (Feet)': 'X', 'Rink Location Y (Feet)': 'Y'})

    # --- THE OT FIX: Clean the Period column before doing math ---
    df_raw['Period'] = df_raw['Period'].apply(extract_period_number)

    def parse_clock(c):
        try:
            m, s = str(c).strip().split(':')
            return int(m) * 60 + int(s)
        except:
            return np.nan

    clock_col = 'Game Clock' if 'Game Clock' in df_raw.columns else 'Clock'
    if clock_col in df_raw.columns:
        df_raw['Seconds_Remaining'] = df_raw[clock_col].apply(parse_clock)
        df_raw['Period_Elapsed'] = 1200 - df_raw['Seconds_Remaining']
        df_raw['True_Game_Elapsed'] = (df_raw['Period'] - 1) * 1200 + df_raw['Period_Elapsed']
    else:
        df_raw['True_Game_Elapsed'] = np.nan

    match = re.search(r'(Team\.[A-Z]).*@.*(Team\.[A-Z])', filename, re.IGNORECASE)
    if match:
        away_team = match.group(1).replace('.', ' ')
        home_team = match.group(2).replace('.', ' ')
    else:
        away_team, home_team = "Away", "Home"

    puck_df = df_raw[df_raw['Player or Puck'] == 'Puck'][['Period', 'Frame_Number', 'X', 'Y']]
    puck_df = puck_df.rename(columns={'X': 'Puck_X', 'Y': 'Puck_Y'})

    df = df_raw[df_raw['Player or Puck'] == 'Player'].copy()
    df = df[df['Player Jersey Number'] != 'Go'].dropna(subset=['Player Jersey Number'])

    team_mapping = {'Away': away_team, 'Home': home_team}
    df['Actual_Team'] = df['Team'].map(team_mapping).fillna(df['Team'])
    df['Unique_Player_ID'] = df['Actual_Team'] + '_#' + df['Player Jersey Number'].astype(int).astype(str)
    df['Game_ID'] = game_date + "_" + away_team + "_at_" + home_team

    df = df.merge(puck_df, on=['Period', 'Frame_Number'], how='inner')
    df = df.sort_values(['Unique_Player_ID', 'Period', 'Frame_Number']).reset_index(drop=True)

    del df_raw, puck_df
    gc.collect()

    team_centroids = df.groupby(['Period', 'Frame_Number', 'Team'])[['X', 'Y']].transform('mean')
    df['Dist_to_Centroid'] = np.hypot(df['X'] - team_centroids['X'], df['Y'] - team_centroids['Y'])
    df['Dist_to_Net'] = np.minimum(np.hypot(df['X'] - 89, df['Y']), np.hypot(df['X'] + 89, df['Y']))
    df['In_Crease'] = (df['Dist_to_Net'] < 8.0).astype(int)
    df['In_Slot'] = (df['Dist_to_Net'] < 20.0).astype(int)
    df['Dist_to_Puck'] = np.hypot(df['X'] - df['Puck_X'], df['Y'] - df['Puck_Y'])
    df['Around_Puck'] = (df['Dist_to_Puck'] < 5.0).astype(int)

    group = df.groupby(['Unique_Player_ID', 'Period'])
    df['Frame_Diff'] = group['Frame_Number'].diff()
    dt_series = df['Frame_Diff'] * DT

    df['Velocity'] = np.hypot(group['X'].diff(), group['Y'].diff()) / dt_series
    df['Velocity'] = group['Velocity'].transform(lambda x: x.ffill().rolling(9, center=True, min_periods=1).mean())
    df.loc[(df['Velocity'] >= MAX_VELOCITY) | (df['Velocity'] <= 0), 'Velocity'] = np.nan

    df['Acceleration'] = (group['Velocity'].diff() / dt_series)
    df.loc[(df['Acceleration'] >= MAX_ACCELERATION) | (df['Acceleration'] <= MAX_DECELERATION), 'Acceleration'] = np.nan

    df['Abs_Acceleration'] = df['Acceleration'].abs()
    df['Abs_Acceleration'] = df.groupby(['Unique_Player_ID', 'Period'])['Abs_Acceleration'].transform(lambda x: x.ffill().rolling(9, center=True, min_periods=1).mean())
    df['Jerk'] = (df.groupby(['Unique_Player_ID', 'Period'])['Abs_Acceleration'].diff() / dt_series).abs()
    df['Jerk'] = df.groupby(['Unique_Player_ID', 'Period'])['Jerk'].transform(lambda x: x.ffill().rolling(9, center=True, min_periods=1).mean())

    df['Closing_Speed_on_Puck'] = -(group['Dist_to_Puck'].diff() / dt_series)

    # Replaced inplace=True to be fully compliant with Pandas 3.0 standards
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=['Velocity', 'Abs_Acceleration', 'Jerk', 'Closing_Speed_on_Puck']).copy()

    df['Time_Bucket'] = df['Frame_Number'] // DOWNSAMPLE_RATE
    ts_df = df.groupby(['Game_ID', 'Actual_Team', 'Unique_Player_ID', 'Period', 'Time_Bucket']).mean(numeric_only=True).reset_index()

    return ts_df

# ==========================================
# 2. STATE CLUSTERING & ACTIVATION SCORING
# ==========================================
def generate_activation_series(master_ts_df):
    print(f"\nClustering {len(master_ts_df)} micro-states across all players...")

    features = [
        'Velocity', 'Abs_Acceleration', 'Jerk',
        'Dist_to_Puck', 'Closing_Speed_on_Puck',
        'Dist_to_Centroid', 'Dist_to_Net', 'In_Crease',
        'In_Slot', 'Around_Puck'
    ]

    X = master_ts_df[features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
    master_ts_df['State_Cluster'] = kmeans.fit_predict(X_scaled)

    # --- RESTORED: Output Cluster Profiles for the "Eye Test" ---
    cluster_means = master_ts_df.groupby('State_Cluster')[features].mean()
    print("\n=== CLUSTER PROFILES ===")
    print(cluster_means.round(2))

    plt.figure(figsize=(12, 6))
    scaled_means = pd.DataFrame(scaler.fit_transform(cluster_means), columns=features, index=cluster_means.index)
    sns.heatmap(scaled_means, annot=True, cmap='coolwarm', center=0)
    plt.title('Micro-State Tactical Heatmap')
    plt.ylabel('Cluster ID')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'Micro_State_Profiles.png'))
    plt.close()
    print(f"\nSaved cluster heatmap to {OUTPUT_DIR}/Micro_State_Profiles.png")
    # -------------------------------------------------------------

    master_ts_df['Is_Active'] = master_ts_df['State_Cluster'].isin(ACTIVE_CLUSTERS).astype(int)

    return master_ts_df

# ==========================================
# 3. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    tracking_files = glob.glob(os.path.join(BASE_DATA_DIR, "**", "*Tracking_P*.csv"), recursive=True)

    # Also grab OT files explicitly if they exist
    ot_files = glob.glob(os.path.join(BASE_DATA_DIR, "**", "*Tracking_POT*.csv"), recursive=True)
    all_files = tracking_files + ot_files

    all_time_series = []

    for f in all_files:
        print(f"Processing Series: {os.path.basename(f)}")
        try:
            ts_data = extract_time_series_features(f)
            if not ts_data.empty:
                all_time_series.append(ts_data)
            del ts_data
            gc.collect()
        except Exception as e:
            print(f"  Error processing {os.path.basename(f)}: {e}")

    if all_time_series:
        master_df = pd.concat(all_time_series)
        final_ts_dataset = generate_activation_series(master_df)

        output_path = os.path.join(OUTPUT_DIR, 'Final_Player_Activation_Data.csv')
        final_ts_dataset = final_ts_dataset.sort_values(['Game_ID', 'Unique_Player_ID', 'True_Game_Elapsed'])
        final_ts_dataset.to_csv(output_path, index=False)

        print(f"\nSUCCESS! Multi-game Time Series saved to {output_path}")

def find_optimal_k(master_ts_df, max_k=10):
    print(f"\nRunning Elbow Method for k=2 to k={max_k}...")

    # Don't forget to include your new feature here!
    features = [
        'Velocity', 'Abs_Acceleration', 'Jerk',
        'Dist_to_Puck', 'Closing_Speed_on_Puck',
        'Dist_to_Centroid', 'Dist_to_Net', 'In_Crease',
        'In_Slot', 'Around_Puck'
    ]

    X = master_ts_df[features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    inertias = []
    K_range = range(2, max_k + 1)

    for k in K_range:
        # n_init='auto' speeds up the process significantly for large datasets
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        kmeans.fit(X_scaled)
        inertias.append(kmeans.inertia_)
        print(f"  k={k} calculated. Inertia: {kmeans.inertia_}")

    # Plot the Elbow Curve
    plt.figure(figsize=(8, 5))
    plt.plot(K_range, inertias, marker='o', linestyle='--', color='b')
    plt.title('Elbow Method For Optimal k (Micro-States)')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia (Sum of Squared Distances)')
    plt.xticks(K_range)
    plt.grid(True)

    output_file = os.path.join(OUTPUT_DIR, 'Elbow_Curve.png')
    plt.savefig(output_file)
    plt.close()

    print(f"\nSaved Elbow Curve to {output_file}")



In [ ]:
# describing teams based on their clusters:
import pandas as pd
import os

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
TRACKING_FILE = os.path.join(BASE_DATA_DIR, "Cluster_Time_Series_Activation/Final_Player_Activation_Data.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_team_cluster_composition():
    print("Loading Master Tracking Data...")
    df = pd.read_csv(TRACKING_FILE, low_memory=False)

    # Drop rows missing team or cluster data just to be safe
    df = df.dropna(subset=['Actual_Team', 'State_Cluster'])

    # Clean team names so they match cleanly
    df['Actual_Team'] = df['Actual_Team'].str.replace('.', ' ', regex=False).str.strip()

    print("Calculating overall time spent in each tactical state...")

    # 1. Count the number of half-second frames spent in each cluster per team
    cluster_counts = df.groupby(['Actual_Team', 'State_Cluster']).size().unstack(fill_value=0)

    # 2. Convert raw counts to percentages of total time
    team_composition = cluster_counts.div(cluster_counts.sum(axis=1), axis=0) * 100

    # 3. Rename columns for a clean table output
    cluster_labels = {
        0: 'C0_Slot_%',
        1: 'C1_Battle_%',
        2: 'C2_Glide_%',
        3: 'C3_Possess_%',
        4: 'C4_Perimeter_%',
        5: 'C5_Crease_%'
    }
    team_composition = team_composition.rename(columns=cluster_labels)
    team_composition = team_composition.round(2)

    # Add a "Total_Active_%" column (sum of 0, 1, 3, 5)
    team_composition['Total_Active_%'] = team_composition[['C0_Slot_%', 'C1_Battle_%', 'C3_Possess_%', 'C5_Crease_%']].sum(axis=1).round(2)

    # Sort by Total Active to see who the highest-tempo teams are
    team_composition = team_composition.sort_values(by='Total_Active_%', ascending=False)

    print("\n=== MACRO TACTICAL COMPOSITION (Baseline System) ===")
    print(team_composition.to_string())

    out_csv = os.path.join(OUTPUT_DIR, "Team_Overall_Cluster_Composition.csv")
    team_composition.to_csv(out_csv)
    print(f"\nSaved composition table to: {out_csv}")

if __name__ == "__main__":
    generate_team_cluster_composition()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
TRACKING_FILE = os.path.join(BASE_DATA_DIR, "Cluster_Time_Series_Activation/Final_Player_Activation_Data.csv")
EVENT_FILES_PATTERN = os.path.join(BASE_DATA_DIR, "Team*@Team*", "*[Ee]vents*.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_BEFORE = 20

def extract_period_number(p_val):
    val_str = str(p_val).upper().strip()
    if val_str in ['OT', 'POT']: return 4
    try:
        match = re.search(r'\d+', val_str)
        return int(match.group()) if match else 1
    except:
        return 1

def clock_to_elapsed(clock_str, period):
    try:
        p = extract_period_number(period)
        parts = str(clock_str).strip().split(':')
        mm, ss = int(parts[0]), int(parts[1])
        if p > 3 and mm < 20:
            return ((p - 1) * 1200) + ((mm * 60 + ss) if mm == 0 and ss == 0 else (300 - (mm * 60 + ss)))
        return ((p - 1) * 1200) + (1200 - (mm * 60 + ss))
    except:
        return np.nan

def create_active_divergence_plots(trend_df, window_size):
    """Plots the total activation trajectory of Goals vs Shots for Both Teams."""
    df = trend_df[trend_df['Rel_Time'] >= -window_size].dropna().copy()
    if df.empty: return

    # Separate our two cohorts
    goals = df[df['Event_Type'] == 'Goal']
    shots = df[df['Event_Type'] == 'Shot']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    fig.suptitle(f'The "Effort" Myth: {window_size}-Second Total Activation (Goals vs. Shots)', fontsize=18, fontweight='bold')

    # --- Subplot 1: Attacking Team ---
    ax = axes[0]
    ax.plot(goals['Rel_Time'], goals['Atk_Smooth'], color='blue', linewidth=4, alpha=0.9, label='Result: GOAL')
    ax.plot(shots['Rel_Time'], shots['Atk_Smooth'], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')

    ax.set_title("Attacking Team Total Activation\n(Not Statistically Significant)", fontsize=14, fontweight='bold')
    ax.set_xlabel('Time to Event (Seconds)', fontsize=12)
    ax.set_ylabel('Average Active Players', fontsize=12)
    ax.axvline(0, color='black', linestyle=':', linewidth=2)
    ax.set_xlim(-window_size, 0)
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(alpha=0.4, linestyle='--')

    # --- Subplot 2: Defending Team ---
    ax = axes[1]
    ax.plot(goals['Rel_Time'], goals['Def_Smooth'], color='red', linewidth=4, alpha=0.9, label='Result: GOAL')
    ax.plot(shots['Rel_Time'], shots['Def_Smooth'], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')

    ax.set_title("Defending Team Total Activation\n(Not Statistically Significant)", fontsize=14, fontweight='bold')
    ax.set_xlabel('Time to Event (Seconds)', fontsize=12)
    ax.set_ylabel('Average Active Players', fontsize=12)
    ax.axvline(0, color='black', linestyle=':', linewidth=2)
    ax.set_xlim(-window_size, 0)
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(alpha=0.4, linestyle='--')

    plt.tight_layout()
    plt.show()
    plot_path = os.path.join(OUTPUT_DIR, f"Active_Players_Trajectory_{window_size}s.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()

    print(f" -> Generated {window_size}-second Goal vs. Shot Total Activation plot.")

def run_total_activation_trajectories():
    print("Loading Master Tracking Data...")
    master_df = pd.read_csv(TRACKING_FILE, low_memory=False)
    master_df['Period'] = master_df['Period'].apply(extract_period_number)

    if 'True_Game_Elapsed' in master_df.columns:
        master_df['elapsed_seconds'] = master_df['True_Game_Elapsed']
        master_df = master_df.dropna(subset=['elapsed_seconds'])
    else:
        print(" [WARNING] True_Game_Elapsed missing.")
        return

    # --- DEFINE TOTAL ACTIVATION ---
    # According to your earlier script, Is_Active combines C0, C1, C3, and C5
    master_df['Is_Active'] = master_df['State_Cluster'].isin([0, 1, 3, 5]).astype(int)

    event_files = glob.glob(EVENT_FILES_PATTERN)
    print(f"Found {len(event_files)} event files. Extracting Goal AND Shot Activation Trajectories...")

    trajectory_data = []

    for event_path in event_files:
        folder_name = os.path.basename(os.path.dirname(event_path))
        events_df = pd.read_csv(event_path)
        clock_col = 'Clock' if 'Clock' in events_df.columns else 'Game Clock'
        if clock_col not in events_df.columns: continue

        events_df['Period'] = events_df['Period'].apply(extract_period_number)
        events_df['elapsed_seconds'] = events_df.apply(lambda r: clock_to_elapsed(r[clock_col], r['Period']), axis=1)

        folder_match = re.search(r'Team[\.\s]*([A-Z])\s*@\s*Team[\.\s]*([A-Z])', folder_name, re.IGNORECASE)
        if not folder_match: continue

        away_team = f"Team {folder_match.group(1).upper()}"
        home_team = f"Team {folder_match.group(2).upper()}"

        mask = master_df['Game_ID'].str.contains(away_team) & master_df['Game_ID'].str.contains(home_team)
        game_tracking = master_df[mask].copy()
        if game_tracking.empty: continue

        # Group by total active players instead of specific clusters
        heartbeat = game_tracking.groupby(['elapsed_seconds', 'Actual_Team'])['Is_Active'].sum().unstack().fillna(0)

        # TARGET BOTH GOALS AND SHOTS
        target_events = events_df[events_df['Event'].str.contains('Goal|Shot', case=False, na=False)]

        for idx, event in target_events.iterrows():
            t = event['elapsed_seconds']
            if pd.isna(t): continue

            # Determine if this row is a Goal or a Shot
            event_type = 'Goal' if 'Goal' in str(event['Event']).title() else 'Shot'

            atk_raw = str(event['Team']).strip()
            team_letter = re.search(r'([A-Z])$', atk_raw)
            atk_team = f"Team {team_letter.group(1).upper()}" if team_letter else atk_raw

            # Determine Defending Team
            if atk_team == home_team: def_team = away_team
            elif atk_team == away_team: def_team = home_team
            else:
                teams_on_ice = [col for col in heartbeat.columns if 'Team' in col]
                def_team = next((team for team in teams_on_ice if team != atk_team), None)

            if atk_team in heartbeat.columns and def_team in heartbeat.columns:
                window = heartbeat[(heartbeat.index >= t - WINDOW_BEFORE) & (heartbeat.index <= t)].copy()

                if not window.empty:
                    for time_index, row in window.iterrows():
                        rel_time = np.round(time_index - t, 1)
                        trajectory_data.append({
                            'Game': folder_name,
                            'Event_ID': f"{folder_name}_{event_type}_{idx}",
                            'Event_Type': event_type,
                            'Rel_Time': rel_time,
                            'Atk_Active': row[atk_team],
                            'Def_Active': row[def_team]
                        })

    traj_df = pd.DataFrame(trajectory_data)
    if traj_df.empty:
        print("No trajectory data found.")
        return

    # Group by BOTH Event_Type and Rel_Time
    trend_df = traj_df.groupby(['Event_Type', 'Rel_Time'])[['Atk_Active', 'Def_Active']].mean().reset_index()

    # Apply rolling average per event type
    def smooth_group(group):
        group['Atk_Smooth'] = group['Atk_Active'].rolling(3, center=True, min_periods=1).mean()
        group['Def_Smooth'] = group['Def_Active'].rolling(3, center=True, min_periods=1).mean()
        return group

    # Apply the smoothing function separately to Goals and Shots
    trend_df = trend_df.groupby('Event_Type', group_keys=False).apply(smooth_group)

    print("\nGenerating Visualizations...")
    create_active_divergence_plots(trend_df, window_size=20)
    create_active_divergence_plots(trend_df, window_size=10)
    create_active_divergence_plots(trend_df, window_size=5)

    print("\n=== SUCCESS: TOTAL ACTIVATION TRAJECTORY ANALYSIS COMPLETE ===")

if __name__ == "__main__":
    run_total_activation_trajectories()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
TRACKING_FILE = os.path.join(BASE_DATA_DIR, "Cluster_Time_Series_Activation/Final_Player_Activation_Data.csv")
EVENT_FILES_PATTERN = os.path.join(BASE_DATA_DIR, "Team*@Team*", "*[Ee]vents*.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")

# Define the three time windows to analyze simultaneously
WINDOWS = [20, 10, 5, 2, 1]

def extract_period_number(p_val):
    val_str = str(p_val).upper().strip()
    if val_str in ['OT', 'POT']: return 4
    try:
        match = re.search(r'\d+', val_str)
        return int(match.group()) if match else 1
    except:
        return 1

def clock_to_elapsed(clock_str, period):
    try:
        p = extract_period_number(period)
        parts = str(clock_str).strip().split(':')
        mm, ss = int(parts[0]), int(parts[1])
        if p > 3 and mm < 20:
            return ((p - 1) * 1200) + ((mm * 60 + ss) if mm == 0 and ss == 0 else (300 - (mm * 60 + ss)))
        return ((p - 1) * 1200) + (1200 - (mm * 60 + ss))
    except:
        return np.nan

def run_multi_window_alignment():
    print("Loading Master Tracking Data...")
    master_df = pd.read_csv(TRACKING_FILE, low_memory=False)
    master_df['Period'] = master_df['Period'].apply(extract_period_number)

    if 'True_Game_Elapsed' in master_df.columns:
        master_df['elapsed_seconds'] = master_df['True_Game_Elapsed']
        master_df = master_df.dropna(subset=['elapsed_seconds'])
    else:
        print(" [WARNING] True_Game_Elapsed missing.")
        return

    # Create distinct flags for our clusters and total activation
    master_df['Is_C0_Slot'] = (master_df['State_Cluster'] == 0).astype(int)
    master_df['Is_C1_Battle'] = (master_df['State_Cluster'] == 1).astype(int)
    master_df['Is_C3_Possess'] = (master_df['State_Cluster'] == 3).astype(int)
    master_df['Is_C5_Crease'] = (master_df['State_Cluster'] == 5).astype(int)
    # Ensure Is_Active is present
    master_df['Is_Active'] = master_df['State_Cluster'].isin([0, 1, 3, 5]).astype(int)

    event_files = glob.glob(EVENT_FILES_PATTERN)
    print(f"Found {len(event_files)} event files. Extracting 20s, 10s, and 5s Profiles...")

    all_game_stats = []

    for event_path in event_files:
        folder_name = os.path.basename(os.path.dirname(event_path))
        events_df = pd.read_csv(event_path)
        clock_col = 'Clock' if 'Clock' in events_df.columns else 'Game Clock'
        if clock_col not in events_df.columns: continue

        events_df['Period'] = events_df['Period'].apply(extract_period_number)
        events_df['elapsed_seconds'] = events_df.apply(lambda r: clock_to_elapsed(r[clock_col], r['Period']), axis=1)

        folder_match = re.search(r'Team[\.\s]*([A-Z])\s*@\s*Team[\.\s]*([A-Z])', folder_name, re.IGNORECASE)
        if not folder_match: continue

        away_team = f"Team {folder_match.group(1).upper()}"
        home_team = f"Team {folder_match.group(2).upper()}"

        mask = master_df['Game_ID'].str.contains(away_team) & master_df['Game_ID'].str.contains(home_team)
        game_tracking = master_df[mask].copy()
        if game_tracking.empty: continue

        # Group by our cluster flags AND the total active flag
        agg_cols = ['Is_C0_Slot', 'Is_C1_Battle', 'Is_C3_Possess', 'Is_C5_Crease', 'Is_Active']
        heartbeat = game_tracking.groupby(['elapsed_seconds', 'Actual_Team'])[agg_cols].sum().unstack().fillna(0)

        target_events = events_df[events_df['Event'].str.contains('Goal|Shot', case=False, na=False)]

        for _, event in target_events.iterrows():
            t = event['elapsed_seconds']
            if pd.isna(t): continue

            atk_raw = str(event['Team']).strip()
            team_letter = re.search(r'([A-Z])$', atk_raw)
            atk_team = f"Team {team_letter.group(1).upper()}" if team_letter else atk_raw

            if atk_team == home_team: def_team = away_team
            elif atk_team == away_team: def_team = home_team
            else:
                teams_on_ice = [col for col in heartbeat['Is_Active'].columns if 'Team' in col]
                def_team = next((team for team in teams_on_ice if team != atk_team), None)

            if atk_team in heartbeat['Is_Active'].columns and def_team in heartbeat['Is_Active'].columns:

                # Base record for the event
                record = {
                    'Game': folder_name,
                    'Event': event['Event'],
                    'Attacking_Team': atk_team,
                    'Defending_Team': def_team,
                    'Time_Elapsed': t
                }

                # Loop through 20, 10, and 5 second windows dynamically
                # Loop through 20, 10, and 5 second windows dynamically
                for w in WINDOWS:
                    window = heartbeat[(heartbeat.index >= t - w) & (heartbeat.index <= t)]

                    if not window.empty:
                        # 1. Total Active Players (Mean over the window)
                        atk_mean = window['Is_Active'][atk_team].mean()
                        def_mean = window['Is_Active'][def_team].mean()
                        record[f'Atk_Active_{w}s'] = round(atk_mean, 2)
                        record[f'Def_Active_{w}s'] = round(def_mean, 2)
                        record[f'Activation_Gap_{w}s'] = round(atk_mean - def_mean, 2)

                        # 2. Peak Attacker Cluster Involvement (Max over the window)
                        record[f'Atk_C0_Slot_{w}s'] = window['Is_C0_Slot'][atk_team].mean()
                        record[f'Atk_C1_Battle_{w}s'] = window['Is_C1_Battle'][atk_team].mean()
                        record[f'Atk_C3_Possess_{w}s'] = window['Is_C3_Possess'][atk_team].mean()
                        record[f'Atk_C5_Crease_{w}s'] = window['Is_C5_Crease'][atk_team].mean()

                        # 3. Peak Defender Cluster Involvement (Max over the window)
                        record[f'Def_C0_Slot_{w}s'] = window['Is_C0_Slot'][def_team].mean()
                        record[f'Def_C1_Battle_{w}s'] = window['Is_C1_Battle'][def_team].mean()
                        record[f'Def_C3_Possess_{w}s'] = window['Is_C3_Possess'][def_team].mean()
                        record[f'Def_C5_Crease_{w}s'] = window['Is_C5_Crease'][def_team].mean()

                # Only append if we successfully got data
                if f'Activation_Gap_{WINDOWS[0]}s' in record:
                    all_game_stats.append(record)

    results_df = pd.DataFrame(all_game_stats)
    out_path = os.path.join(OUTPUT_DIR, "Multi_Window_Tactical_Alignment.csv")
    results_df.to_csv(out_path, index=False)

    print(f"\n=== SUCCESS: MULTI-WINDOW PROFILES SAVED TO {out_path} ===")

if __name__ == "__main__":
    run_multi_window_alignment()

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
ALIGNMENT_FILE = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports/Multi_Window_Tactical_Alignment.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")

WINDOWS = [20, 10, 5, 2, 1]

def run_multi_window_statistics():
    print("Loading Multi-Window Tactical Data...")
    df = pd.read_csv(ALIGNMENT_FILE)

    df['Event_Type'] = df['Event'].apply(lambda x: 'Goal' if 'Goal' in str(x).title() else 'Shot')

    goals = df[df['Event_Type'] == 'Goal']
    shots = df[df['Event_Type'] == 'Shot']

    print(f"\nSample Sizes -> Goals: {len(goals)} | Shots: {len(shots)}")

    for w in WINDOWS:
        print(f"\n{'='*40}")
        print(f"       {w}-SECOND WINDOW ANALYSIS")
        print(f"{'='*40}")

        # Define the metrics we generated for this specific window
        # Define ALL the metrics we generated for this specific window
        metrics = [
            f'Atk_Active_{w}s', f'Def_Active_{w}s', f'Activation_Gap_{w}s',
            f'Atk_C0_Slot_{w}s', f'Atk_C1_Battle_{w}s', f'Atk_C3_Possess_{w}s', f'Atk_C5_Crease_{w}s',
            f'Def_C0_Slot_{w}s', f'Def_C1_Battle_{w}s', f'Def_C3_Possess_{w}s', f'Def_C5_Crease_{w}s'
        ]

        for metric in metrics:
            if metric not in df.columns: continue

            g_val = goals[metric].dropna()
            s_val = shots[metric].dropna()

            t_stat, p_val = stats.ttest_ind(g_val, s_val, equal_var=False)

            if p_val < 0.05:
                print(f"✅ SIGNIFICANT: {metric}")
                print(f"   Goals Mean: {g_val.mean():.2f} | Shots Mean: {s_val.mean():.2f}")
                print(f"   P-Value:    {p_val:.4f}\n")
            else:
                print(f"❌ Not Sig: {metric} (P-Value: {p_val:.4f})")

    # Generate a plot for the 5-second Activation Gap
    if 'Activation_Gap_5s' in df.columns:
        plt.figure(figsize=(9, 6))
        sns.boxplot(x='Event_Type', y='Activation_Gap_5s', hue='Event_Type', data=df,
                    palette={'Goal': 'gold', 'Shot': 'lightgrey'}, legend=False, width=0.5,
                    showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})

        sns.stripplot(x='Event_Type', y='Activation_Gap_5s', hue='Event_Type', data=df,
                      palette={'Goal': 'darkgoldenrod', 'Shot': 'dimgrey'}, legend=False,
                      alpha=0.5, jitter=True, size=5)

        plt.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.6)
        plt.text(0.5, 0.1, "Offense Dominating", color='red', fontsize=10)

        plt.title("The Activation Gap (5s Lead-up): Goals vs. Shots", fontsize=14, fontweight='bold')
        plt.xlabel("Event Outcome", fontsize=12)
        plt.ylabel("Gap (Attacking Active - Defending Active)", fontsize=12)
        plt.grid(axis='y', linestyle=':', alpha=0.7)

        plot_path = os.path.join(OUTPUT_DIR, "Activation_Gap_5s_Stats.png")
        plt.savefig(plot_path, dpi=150)
        plt.close()
        print(f"\nSaved gap plot to {plot_path}")

if __name__ == "__main__":
    run_multi_window_statistics()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
TRACKING_FILE = os.path.join(BASE_DATA_DIR, "Cluster_Time_Series_Activation/Final_Player_Activation_Data.csv")
EVENT_FILES_PATTERN = os.path.join(BASE_DATA_DIR, "Team*@Team*", "*[Ee]vents*.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_BEFORE = 20

def extract_period_number(p_val):
    val_str = str(p_val).upper().strip()
    if val_str in ['OT', 'POT']: return 4
    try:
        match = re.search(r'\d+', val_str)
        return int(match.group()) if match else 1
    except:
        return 1

def clock_to_elapsed(clock_str, period):
    try:
        p = extract_period_number(period)
        parts = str(clock_str).strip().split(':')
        mm, ss = int(parts[0]), int(parts[1])
        if p > 3 and mm < 20:
            return ((p - 1) * 1200) + ((mm * 60 + ss) if mm == 0 and ss == 0 else (300 - (mm * 60 + ss)))
        return ((p - 1) * 1200) + (1200 - (mm * 60 + ss))
    except:
        return np.nan

def create_divergence_plots(trend_df, window_size):
    """Plots the trajectory of Goals vs Shots for the significant clusters."""
    df = trend_df[trend_df['Rel_Time'] >= -window_size].dropna().copy()
    if df.empty: return

    # Separate our two cohorts
    goals = df[df['Event_Type'] == 'Goal']
    shots = df[df['Event_Type'] == 'Shot']

    # We will plot the two clusters that proved statistically significant
    clusters_to_plot = {
        'C0_Slot': {'col': 'C0_Smooth', 'color': '#FF0000', 'title': 'High Slot Profiles (C0)'},
        'C3_Possess': {'col': 'C3_Smooth', 'color': '#00BFFF', 'title': 'Puck Possession Profiles (C3)'}
    }

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'The Anatomy of Conversion: {window_size}-Second Trajectories Leading to Event', fontsize=18, fontweight='bold')

    for i, (key, props) in enumerate(clusters_to_plot.items()):
        ax = axes[i]

        # Plot Goals (Solid, bold line)
        ax.plot(goals['Rel_Time'], goals[props['col']], color=props['color'], linewidth=4, alpha=0.9, label='Result: GOAL')

        # Plot Shots (Dashed, thinner line)
        ax.plot(shots['Rel_Time'], shots[props['col']], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')

        ax.set_title(props['title'], fontsize=14, fontweight='bold')
        ax.set_xlabel('Time to Event (Seconds)', fontsize=12)
        ax.set_ylabel('Average Attacking Players in State', fontsize=12)
        ax.axvline(0, color='black', linestyle=':', linewidth=2)
        ax.set_xlim(-window_size, 0)
        ax.legend(fontsize=11, loc='upper left')
        ax.grid(alpha=0.4, linestyle='--')

    plt.tight_layout()
    plt.show()
    plot_path = os.path.join(OUTPUT_DIR, f"Divergence_C0_C3_Trajectories_{window_size}s.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()

    print(f" -> Generated {window_size}-second Goal vs. Shot Divergence plot.")

def run_cluster_trajectories():
    print("Loading Master Tracking Data...")
    master_df = pd.read_csv(TRACKING_FILE, low_memory=False)
    master_df['Period'] = master_df['Period'].apply(extract_period_number)

    if 'True_Game_Elapsed' in master_df.columns:
        master_df['elapsed_seconds'] = master_df['True_Game_Elapsed']
        master_df = master_df.dropna(subset=['elapsed_seconds'])
    else:
        print(" [WARNING] True_Game_Elapsed missing.")
        return

    # --- ISOLATE CLUSTERS ---
    master_df['Is_C0_Slot'] = (master_df['State_Cluster'] == 0).astype(int)
    master_df['Is_C1_Battle'] = (master_df['State_Cluster'] == 1).astype(int)
    master_df['Is_C2_Transition'] = (master_df['State_Cluster'] == 2).astype(int)
    master_df['Is_C3_Possess'] = (master_df['State_Cluster'] == 3).astype(int)
    master_df['Is_C4_Floater'] = (master_df['State_Cluster'] == 4).astype(int)
    master_df['Is_C5_Crease'] = (master_df['State_Cluster'] == 5).astype(int)

    event_files = glob.glob(EVENT_FILES_PATTERN)
    print(f"Found {len(event_files)} event files. Extracting 20-second Goal AND Shot Trajectories...")

    trajectory_data = []

    for event_path in event_files:
        folder_name = os.path.basename(os.path.dirname(event_path))
        events_df = pd.read_csv(event_path)
        clock_col = 'Clock' if 'Clock' in events_df.columns else 'Game Clock'
        if clock_col not in events_df.columns: continue

        events_df['Period'] = events_df['Period'].apply(extract_period_number)
        events_df['elapsed_seconds'] = events_df.apply(lambda r: clock_to_elapsed(r[clock_col], r['Period']), axis=1)

        folder_match = re.search(r'Team[\.\s]*([A-Z])\s*@\s*Team[\.\s]*([A-Z])', folder_name, re.IGNORECASE)
        if not folder_match: continue

        away_team = f"Team {folder_match.group(1).upper()}"
        home_team = f"Team {folder_match.group(2).upper()}"

        mask = master_df['Game_ID'].str.contains(away_team) & master_df['Game_ID'].str.contains(home_team)
        game_tracking = master_df[mask].copy()
        if game_tracking.empty: continue

        heartbeat = game_tracking.groupby(['elapsed_seconds', 'Actual_Team'])[['Is_C0_Slot', 'Is_C1_Battle','Is_C2_Transition', 'Is_C3_Possess', 'Is_C4_Floater', 'Is_C5_Crease']].sum().unstack().fillna(0)

        # TARGET BOTH GOALS AND SHOTS
        target_events = events_df[events_df['Event'].str.contains('Goal|Shot', case=False, na=False)]

        for idx, event in target_events.iterrows():
            t = event['elapsed_seconds']
            if pd.isna(t): continue

            # Determine if this row is a Goal or a Shot
            event_type = 'Goal' if 'Goal' in str(event['Event']).title() else 'Shot'

            atk_raw = str(event['Team']).strip()
            team_letter = re.search(r'([A-Z])$', atk_raw)
            atk_team = f"Team {team_letter.group(1).upper()}" if team_letter else atk_raw

            teams_on_ice = [col for col in heartbeat['Is_C0_Slot'].columns if 'Team' in col]
            if atk_team in teams_on_ice:
                window = heartbeat[(heartbeat.index >= t - WINDOW_BEFORE) & (heartbeat.index <= t)].copy()

                if not window.empty:
                    for time_index, row in window.iterrows():
                        rel_time = np.round(time_index - t, 1)
                        trajectory_data.append({
                            'Game': folder_name,
                            'Event_ID': f"{folder_name}_{event_type}_{idx}",
                            'Event_Type': event_type, # <-- NEW CLASSIFIER
                            'Rel_Time': rel_time,
                            'C0_Slot': row['Is_C0_Slot'][atk_team],
                            'C1_Battle': row['Is_C1_Battle'][atk_team],
                            'C2_Transition': row['Is_C2_Transition'][atk_team],
                            'C3_Possess': row['Is_C3_Possess'][atk_team],
                            'C4_Floater': row['Is_C4_Floater'][atk_team],
                            'C5_Crease': row['Is_C5_Crease'][atk_team]
                        })

    traj_df = pd.DataFrame(trajectory_data)
    if traj_df.empty:
        print("No trajectory data found.")
        return

    # Group by BOTH Event_Type and Rel_Time
    trend_df = traj_df.groupby(['Event_Type', 'Rel_Time'])[['C0_Slot', 'C1_Battle', 'C2_Transition', 'C3_Possess', 'C4_Floater', 'C5_Crease']].mean().reset_index()

    # Apply rolling average per event type
    def smooth_group(group):
        clusters = ['C0_Slot', 'C1_Battle', 'C2_Transition', 'C3_Possess', 'C4_Floater', 'C5_Crease']
        for i, col in enumerate(clusters):
            group[f'C{i}_Smooth'] = group[col].rolling(3, center=True, min_periods=1).mean()
        return group

    # Apply the smoothing function separately to Goals and Shots so they don't blur together
    trend_df = trend_df.groupby('Event_Type', group_keys=False).apply(smooth_group)

    print("\nGenerating Visualizations...")
    create_divergence_plots(trend_df, window_size=20)
    create_divergence_plots(trend_df, window_size=10)
    create_divergence_plots(trend_df, window_size=5)

    print("\n=== SUCCESS: DIVERGENCE TRAJECTORY ANALYSIS COMPLETE ===")

if __name__ == "__main__":
    run_cluster_trajectories()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
TRACKING_FILE = os.path.join(BASE_DATA_DIR, "Cluster_Time_Series_Activation/Final_Player_Activation_Data.csv")
EVENT_FILES_PATTERN = os.path.join(BASE_DATA_DIR, "Team*@Team*", "*[Ee]vents*.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_BEFORE = 20  # Extract 20s, but we will strictly plot 5s


def extract_period_number(p_val):
    val_str = str(p_val).upper().strip()
    if val_str in ['OT', 'POT']:
        return 4
    try:
        match = re.search(r'\d+', val_str)
        return int(match.group()) if match else 1
    except:
        return 1


def clock_to_elapsed(clock_str, period):
    try:
        p = extract_period_number(period)
        parts = str(clock_str).strip().split(':')
        mm, ss = int(parts[0]), int(parts[1])
        if p > 3 and mm < 20:
            return ((p - 1) * 1200) + ((mm * 60 + ss) if mm == 0 and ss == 0 else (300 - (mm * 60 + ss)))
        return ((p - 1) * 1200) + (1200 - (mm * 60 + ss))
    except:
        return np.nan


def create_combined_4panel_plot(trend_df, window_size=5):
    """Plots a 2x2 grid of 5-second trajectories for Goals vs Shots."""
    df = trend_df[trend_df['Rel_Time'] >= -window_size].dropna().copy()
    if df.empty:
        return

    goals = df[df['Event_Type'] == 'Goal']
    shots = df[df['Event_Type'] == 'Shot']

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    #fig.suptitle(
    #    f'General Effort vs. Structural Buildup: {window_size}-Second Lead-Up to Event',
    #    fontsize=20, fontweight='bold', y=0.98
    #)

    # ---------------------------------------------------------
    # ROW 1: TOTAL ACTIVATION (Not Significant)
    # ---------------------------------------------------------
    # Top-Left: Attacking Team
    ax1 = axes[0, 0]
    ax1.plot(goals['Rel_Time'], goals['Atk_Active_Smooth'], color='blue', linewidth=4, alpha=0.9, label='Result: GOAL')
    ax1.plot(shots['Rel_Time'], shots['Atk_Active_Smooth'], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')
    ax1.set_title("Attacking Team Total Activation\n(Not Statistically Significant)", fontsize=14, fontweight='bold')
    ax1.set_ylabel('Average Active Players', fontsize=12)
    ax1.axvline(0, color='black', linestyle=':', linewidth=2)
    ax1.set_xlim(-window_size, 0)
    ax1.legend(fontsize=11, loc='upper left')
    ax1.grid(alpha=0.4, linestyle='--')

    # Top-Right: Defending Team
    ax2 = axes[0, 1]
    ax2.plot(goals['Rel_Time'], goals['Def_Active_Smooth'], color='red', linewidth=4, alpha=0.9, label='Result: GOAL')
    ax2.plot(shots['Rel_Time'], shots['Def_Active_Smooth'], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')
    ax2.set_title("Defending Team Total Activation\n(Not Statistically Significant)", fontsize=14, fontweight='bold')
    ax2.set_ylabel('Average Active Players', fontsize=12)
    ax2.axvline(0, color='black', linestyle=':', linewidth=2)
    ax2.set_xlim(-window_size, 0)
    ax2.legend(fontsize=11, loc='upper left')
    ax2.grid(alpha=0.4, linestyle='--')

    # ---------------------------------------------------------
    # ROW 2: SPECIFIC STRUCTURAL STATES
    # ---------------------------------------------------------
    # Bottom-Left: High Slot (C0)
    ax3 = axes[1, 0]
    ax3.plot(goals['Rel_Time'], goals['C0_Smooth'], color='#FF0000', linewidth=4, alpha=0.9, label='Result: GOAL')
    ax3.plot(shots['Rel_Time'], shots['C0_Smooth'], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')
    ax3.set_title("High Slot Profiles (C0)\n(Significant peak: p < 0.05 across all windows)", fontsize=14, fontweight='bold')
    ax3.set_xlabel('Time to Event (Seconds)', fontsize=12)
    ax3.set_ylabel('Average Attacking Players in State', fontsize=12)
    ax3.axvline(0, color='black', linestyle=':', linewidth=2)
    ax3.set_xlim(-window_size, 0)
    ax3.legend(fontsize=11, loc='upper left')
    ax3.grid(alpha=0.4, linestyle='--')

    # Bottom-Right: Deep Crease (C5)
    ax4 = axes[1, 1]
    ax4.plot(goals['Rel_Time'], goals['C5_Smooth'], color='#00BFFF', linewidth=4, alpha=0.9, label='Result: GOAL')
    ax4.plot(shots['Rel_Time'], shots['C5_Smooth'], color='grey', linewidth=2, linestyle='--', alpha=0.8, label='Result: SHOT (No Goal)')
    ax4.set_title("Deep Crease Profiles (C5)\n(Significant at 1s mean: p = 0.008)", fontsize=14, fontweight='bold')
    ax4.set_xlabel('Time to Event (Seconds)', fontsize=12)
    ax4.set_ylabel('Average Attacking Players in State', fontsize=12)
    ax4.axvline(0, color='black', linestyle=':', linewidth=2)
    ax4.set_xlim(-window_size, 0)
    ax4.legend(fontsize=11, loc='upper left')
    ax4.grid(alpha=0.4, linestyle='--')

    panel_labels = ['A)', 'B)', 'C)', 'D)']
    for i, ax in enumerate([ax1, ax2, ax3, ax4]):
        ax.text(-0.08, 1.08, panel_labels[i], transform=ax.transAxes,
                fontsize=16, fontweight='bold', va='bottom', ha='right')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    plot_path = os.path.join(OUTPUT_DIR, f"Combined_4Panel_Trajectories_{window_size}s.png")
    plt.savefig(plot_path, dpi=200)
    plt.close()

    print(f" -> Generated {window_size}-second 4-Panel Divergence Plot.")


def run_unified_trajectories():
    print("Loading Master Tracking Data...")
    master_df = pd.read_csv(TRACKING_FILE, low_memory=False)
    master_df['Period'] = master_df['Period'].apply(extract_period_number)

    if 'True_Game_Elapsed' in master_df.columns:
        master_df['elapsed_seconds'] = master_df['True_Game_Elapsed']
        master_df = master_df.dropna(subset=['elapsed_seconds'])
    else:
        print(" [WARNING] True_Game_Elapsed missing.")
        return

    # --- DEFINE ALL FLAGS ---
    master_df['Is_Active'] = master_df['State_Cluster'].isin([0, 1, 3, 5]).astype(int)
    master_df['Is_C0_Slot'] = (master_df['State_Cluster'] == 0).astype(int)
    master_df['Is_C3_Possess'] = (master_df['State_Cluster'] == 3).astype(int)
    master_df['Is_C5_Crease'] = (master_df['State_Cluster'] == 5).astype(int)

    event_files = glob.glob(EVENT_FILES_PATTERN)
    print(f"Found {len(event_files)} event files. Extracting Unified Trajectories...")

    trajectory_data = []

    for event_path in event_files:
        folder_name = os.path.basename(os.path.dirname(event_path))
        events_df = pd.read_csv(event_path)
        clock_col = 'Clock' if 'Clock' in events_df.columns else 'Game Clock'
        if clock_col not in events_df.columns:
            continue

        events_df['Period'] = events_df['Period'].apply(extract_period_number)
        events_df['elapsed_seconds'] = events_df.apply(
            lambda r: clock_to_elapsed(r[clock_col], r['Period']), axis=1
        )

        folder_match = re.search(r'Team[\.\s]*([A-Z])\s*@\s*Team[\.\s]*([A-Z])', folder_name, re.IGNORECASE)
        if not folder_match:
            continue

        away_team = f"Team {folder_match.group(1).upper()}"
        home_team = f"Team {folder_match.group(2).upper()}"

        mask = master_df['Game_ID'].str.contains(away_team) & master_df['Game_ID'].str.contains(home_team)
        game_tracking = master_df[mask].copy()
        if game_tracking.empty:
            continue

        # Group by all features
        heartbeat = game_tracking.groupby(['elapsed_seconds', 'Actual_Team'])[
            ['Is_Active', 'Is_C0_Slot', 'Is_C3_Possess', 'Is_C5_Crease']
        ].sum().unstack().fillna(0)

        target_events = events_df[events_df['Event'].str.contains('Goal|Shot', case=False, na=False)]

        for idx, event in target_events.iterrows():
            t = event['elapsed_seconds']
            if pd.isna(t):
                continue

            event_type = 'Goal' if 'Goal' in str(event['Event']).title() else 'Shot'

            atk_raw = str(event['Team']).strip()
            team_letter = re.search(r'([A-Z])$', atk_raw)
            atk_team = f"Team {team_letter.group(1).upper()}" if team_letter else atk_raw

            if atk_team == home_team:
                def_team = away_team
            elif atk_team == away_team:
                def_team = home_team
            else:
                teams_on_ice = [col for col in heartbeat['Is_Active'].columns if 'Team' in col]
                def_team = next((team for team in teams_on_ice if team != atk_team), None)

            if atk_team in heartbeat['Is_Active'].columns and def_team in heartbeat['Is_Active'].columns:
                window = heartbeat[(heartbeat.index >= t - WINDOW_BEFORE) & (heartbeat.index <= t)].copy()

                if not window.empty:
                    for time_index, row in window.iterrows():
                        rel_time = np.round(time_index - t, 1)
                        trajectory_data.append({
                            'Game': folder_name,
                            'Event_ID': f"{folder_name}_{event_type}_{idx}",
                            'Event_Type': event_type,
                            'Rel_Time': rel_time,
                            'Atk_Active': row['Is_Active'][atk_team],
                            'Def_Active': row['Is_Active'][def_team],
                            'Atk_C0_Slot': row['Is_C0_Slot'][atk_team],
                            'Atk_C3_Possess': row['Is_C3_Possess'][atk_team],
                            'Atk_C5_Crease': row['Is_C5_Crease'][atk_team],
                        })

    traj_df = pd.DataFrame(trajectory_data)
    if traj_df.empty:
        print("No trajectory data found.")
        return

    # Group by Event_Type and Rel_Time
    trend_df = traj_df.groupby(['Event_Type', 'Rel_Time'])[
        ['Atk_Active', 'Def_Active', 'Atk_C0_Slot', 'Atk_C3_Possess', 'Atk_C5_Crease']
    ].mean().reset_index()

    # Apply rolling average per event type
    def smooth_group(group):
        group['Atk_Active_Smooth'] = group['Atk_Active'].rolling(3, center=True, min_periods=1).mean()
        group['Def_Active_Smooth'] = group['Def_Active'].rolling(3, center=True, min_periods=1).mean()
        group['C0_Smooth'] = group['Atk_C0_Slot'].rolling(3, center=True, min_periods=1).mean()
        group['C3_Smooth'] = group['Atk_C3_Possess'].rolling(3, center=True, min_periods=1).mean()
        group['C5_Smooth'] = group['Atk_C5_Crease'].rolling(3, center=True, min_periods=1).mean()
        return group

    trend_df = trend_df.groupby('Event_Type', group_keys=False).apply(smooth_group)

    print("\nGenerating Visualizations...")
    create_combined_4panel_plot(trend_df, window_size=5)

    print("\n=== SUCCESS: UNIFIED TRAJECTORY ANALYSIS COMPLETE ===")


if __name__ == "__main__":
    run_unified_trajectories()

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/Sports-Analytics/Data/"
TRACKING_FILE = os.path.join(BASE_DATA_DIR, "Cluster_Time_Series_Activation/Final_Player_Activation_Data.csv")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "Game_Analytic_Reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({'font.size': 14})

# Define our cluster names for clean plotting
CLUSTER_LABELS = [
    '0: High Slot',
    '1: Battle/Scrum',
    '2: Transition/Glide',
    '3: Possession',
    '4: Perimeter/Floater',
    '5: Deep Crease'
]

def build_transition_matrices():
    print("Loading Tracking Data for Markov Chains...")
    df = pd.read_csv(TRACKING_FILE, low_memory=False)

    # Ensure we have the exact chronological ordering required for a Markov Chain
    df = df.sort_values(['Game_ID', 'Unique_Player_ID', 'True_Game_Elapsed']).reset_index(drop=True)

    print("Calculating player state transitions...")
    # Shift the cluster column up by 1 to get the "Next State"
    df['Next_State'] = df.groupby(['Game_ID', 'Unique_Player_ID'])['State_Cluster'].shift(-1)

    # Shift the time column up by 1 to get the exact time gap to the next state
    df['Time_to_Next'] = df.groupby(['Game_ID', 'Unique_Player_ID'])['True_Game_Elapsed'].diff().shift(-1)

    # ---------------------------------------------------------
    # THE WHISTLE FILTER:
    # Since our data is 2Hz (0.5s gaps), we drop any transition
    # where the gap is greater than 0.6s (whistles, shifts, etc.)
    # ---------------------------------------------------------
    valid_transitions = df[(df['Time_to_Next'] > 0) & (df['Time_to_Next'] <= 0.6)].copy()

    # Drop NaNs just in case (the last frame for every player will naturally be NaN)
    valid_transitions = valid_transitions.dropna(subset=['Next_State'])

    # Ensure integer types for the crosstab
    valid_transitions['State_Cluster'] = valid_transitions['State_Cluster'].astype(int)
    valid_transitions['Next_State'] = valid_transitions['Next_State'].astype(int)

    # ==========================================
    # 1. THE GLOBAL MATRIX
    # ==========================================
    print("\nGenerating Global Transition Matrix...")
    global_matrix = pd.crosstab(
        valid_transitions['State_Cluster'],
        valid_transitions['Next_State'],
        normalize='index'
    ) * 100 # Convert to percentage (0-100)

    # --- NEW: Save the Raw 6x6 Global Matrix to CSV ---
    clean_global_matrix = global_matrix.copy()
    clean_global_matrix.index = CLUSTER_LABELS   # Name the rows
    clean_global_matrix.columns = CLUSTER_LABELS # Name the columns

    global_csv_path = os.path.join(OUTPUT_DIR, "Markov_Global_Matrix.csv")
    clean_global_matrix.to_csv(global_csv_path)
    print(f" -> Saved Global Matrix CSV to {global_csv_path}")
    # --------------------------------------------------

    # Plot the Global Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(global_matrix, annot=True, fmt=".1f", cmap="YlGnBu",
            xticklabels=CLUSTER_LABELS, yticklabels=CLUSTER_LABELS, 
            vmin=0, vmax=100,
            annot_kws={"size": 14})
    plt.ylabel("Current State ($S_t$)", fontsize=16)
    plt.xlabel("Next State ($S_{t+0.5s}$)", fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize=13)
    plt.yticks(fontsize=13)
    plt.tight_layout()
    
    global_plot_path = os.path.join(OUTPUT_DIR, "Markov_Global_Matrix.png")
    plt.savefig(global_plot_path, dpi=150)
    plt.show()
    plt.close()
    print(f" -> Saved Global Matrix to {global_plot_path}")

    # ==========================================
    # 2. TEAM-SPECIFIC MATRICES
    # ==========================================
    print("\nExtracting Team-Specific Matrices...")
    teams = valid_transitions['Actual_Team'].unique()

    team_data = []

    for team in teams:
        team_df = valid_transitions[valid_transitions['Actual_Team'] == team]

        if team_df.empty: continue

        team_matrix = pd.crosstab(
            team_df['State_Cluster'],
            team_df['Next_State'],
            normalize='index'
        ) * 100

        # Safely extract specific pathways of interest to compare teams
        def get_prob(matrix, start, end):
            try: return round(matrix.loc[start, end], 2)
            except KeyError: return 0.0

        team_data.append({
            'Team': team,
            # The Diagonal: Stickiness
            'Stickiness_C0_Slot': get_prob(team_matrix, 0, 0),
            'Stickiness_C3_Possess': get_prob(team_matrix, 3, 3),

            # The Golden Pathways we identified
            'Pathway_C2_to_C0 (Off-Ball IQ)': get_prob(team_matrix, 2, 0),
            'Pathway_C0_to_C3 (Slot Targeting)': get_prob(team_matrix, 0, 3),
            'Pathway_C1_to_C3 (Puck Recovery)': get_prob(team_matrix, 1, 3)
        })

    # Save the summarized team comparison to CSV
    team_comparison_df = pd.DataFrame(team_data).sort_values('Pathway_C0_to_C3 (Slot Targeting)', ascending=False)
    team_csv_path = os.path.join(OUTPUT_DIR, "Markov_Team_Pathways.csv")
    team_comparison_df.to_csv(team_csv_path, index=False)

    print(f"\n=== SUCCESS: MARKOV CHAIN EXTRACTION COMPLETE ===")
    print("\nTop 5 Teams by Slot Targeting (C0 -> C3):")
    print(team_comparison_df[['Team', 'Pathway_C0_to_C3 (Slot Targeting)']].head(5))

if __name__ == "__main__":
    build_transition_matrices()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def create_tactical_quadrant():
    # Load in the exact data you provided
    data = {
        'Team': ['Team L', 'Team J', 'Team E', 'Team C', 'Team G', 'Team D',
                 'Team K', 'Team F', 'Team I', 'Team H', 'Team A', 'Team B'],
        'C0_to_C3': [3.83, 3.16, 3.14, 2.92, 2.87, 2.70, 2.58, 2.54, 2.50, 2.38, 2.21, 2.06],
        'C1_to_C3': [3.64, 3.81, 4.53, 3.68, 3.70, 4.05, 4.37, 2.50, 3.87, 4.32, 4.20, 5.80]
    }
    df = pd.DataFrame(data)

    plt.figure(figsize=(11, 8))

    # Create the scatter plot
    sns.scatterplot(data=df, x='C1_to_C3', y='C0_to_C3', s=200, color='dodgerblue', edgecolor='black', zorder=5)

    # Calculate medians to draw the crosshairs (Quadrant dividers)
    x_mid = df['C1_to_C3'].median()
    y_mid = df['C0_to_C3'].median()

    plt.axvline(x_mid, color='black', linestyle='--', alpha=0.6, zorder=1)
    plt.axhline(y_mid, color='black', linestyle='--', alpha=0.6, zorder=1)

    # Add Team Labels slightly offset from the dots
    for i in range(df.shape[0]):
        plt.text(df['C1_to_C3'].iloc[i] + 0.05,
                 df['C0_to_C3'].iloc[i] + 0.02,
                 df['Team'].iloc[i],
                 fontsize=11, fontweight='500')

    # Add Quadrant Labels to tell the story
   # Quadrant labels pinned to corners
    plt.text(df['C1_to_C3'].max() + 0.3, df['C0_to_C3'].max() + 0.25,
             "Well-Rounded\n(Wins battles & activates slot)",
             color='green', fontsize=12, alpha=0.7, fontweight='bold',
             ha='right', va='top')

    plt.text(df['C1_to_C3'].min() - 0.3, df['C0_to_C3'].max() + 0.25,
             "Precision Assassins\n(Slot specialists, avoid scrums)",
             color='purple', fontsize=12, alpha=0.7, fontweight='bold',
             ha='left', va='top')

    plt.text(df['C1_to_C3'].max() + 0.3, df['C0_to_C3'].min() - 0.25,
             "The Grinders\n(All hustle, poor slot activation)",
             color='darkorange', fontsize=12, alpha=0.7, fontweight='bold',
             ha='right', va='bottom')

    plt.text(df['C1_to_C3'].min() - 0.3, df['C0_to_C3'].min() - 0.25,
             "Static Systems\n(Low recovery, low activation)",
             color='red', fontsize=12, alpha=0.7, fontweight='bold',
             ha='left', va='bottom')

    #plt.title("Team Tactical Fingerprints: How Teams Acquire Possession", fontsize=16, fontweight='bold')
    plt.xlabel("Explosive Recovery: Battle to Possession (C1 $\\rightarrow$ C3 %)", fontsize=13)
    plt.ylabel("High-Value Acquisition: High Slot to Possession (C0 $\\rightarrow$ C3 %)", fontsize=13)

    # Add a bit of padding to the axes so labels don't get cut off
    plt.xlim(df['C1_to_C3'].min() - 0.4, df['C1_to_C3'].max() + 0.5)
    plt.ylim(df['C0_to_C3'].min() - 0.3, df['C0_to_C3'].max() + 0.3)

    plt.grid(alpha=0.3)
    plt.tight_layout()

    plot_path = os.path.join(OUTPUT_DIR, "Tactical_Quadrant_Scatter.png")
    plt.savefig(plot_path, dpi=150)
    plt.show()

create_tactical_quadrant()